### V0.2

* 5) default  :object	債務不履行があるかどうか（yes / no）　過去にローンやクレジットカードの支払いが滞ったことがあるか
* 6) balance  :int64	    年間の平均残高。
* 7) housing  :object	住宅ローンがあるかどうか（yes / no）
* 8) loan     :object	個人ローンがあるかどうか（yes / no）

# ライブラリ

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

# 学習モデル作成

In [17]:
df = pd.read_pickle('data.pkl')
df

,id,default,balance,housing,loan,y
0,0,no,7,no,no,0
1,1,no,514,no,no,0
2,2,no,602,yes,no,0
3,3,no,34,yes,no,0
4,4,no,889,yes,no,1
...,...,...,...,...,...,...
749995,749995,no,1282,no,yes,1
749996,749996,no,631,no,no,0
749997,749997,no,217,yes,no,0
749998,749998,no,-274,no,no,0


In [18]:
y = df["y"]
x = df.drop(columns=["y", "id"])
x = pd.get_dummies(x, drop_first=True)


In [19]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42)

In [20]:
train_data = lgb.Dataset(x_train,label=y_train)
params = {
    'objective': 'binary',
    'boosting_type': 'gbdt',
    'metric': 'binary_logloss',
    'num_leaves': 31,
    'learning_rate': 0.05    
}

In [21]:
model = lgb.train(params, train_data,num_boost_round=100)

[LightGBM] [Info] Number of positive: 72283, number of negative: 527717
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004946 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 261
[LightGBM] [Info] Number of data points in the train set: 600000, number of used features: 4
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.120472 -> initscore=-1.987971
[LightGBM] [Info] Start training from score -1.987971


In [22]:
y_pred_proba = model.predict(x_test)
y_pred_proba

array([0.07958348, 0.32546649, 0.20754499, ..., 0.12043996, 0.09867133,
       0.11554876], shape=(150000,))

In [23]:
auc = roc_auc_score(y_test, y_pred_proba)
print("roc-auc : ",auc)

roc-auc :  0.7515724085199316


# テスト、提出

In [24]:
df_test = pd.read_pickle('test.pkl')
df_test

,id,default,balance,housing,loan
0,750000,no,1397,yes,no
1,750001,no,23,yes,no
2,750002,no,46,yes,yes
3,750003,no,-1380,yes,yes
4,750004,no,1950,yes,no
...,...,...,...,...,...
249995,999995,no,0,yes,no
249996,999996,no,522,yes,no
249997,999997,no,33,no,no
249998,999998,no,2629,yes,no


In [25]:
submit_x = df_test.drop(columns=["id"])
submit_x = pd.get_dummies(submit_x, drop_first=True)
submit_x = submit_x.reindex(columns=x.columns, fill_value=0)

In [26]:
submit_x

,balance,default_yes,housing_yes,loan_yes
0,1397,False,True,False
1,23,False,True,False
2,46,False,True,True
3,-1380,False,True,True
4,1950,False,True,False
...,...,...,...,...
249995,0,False,True,False
249996,522,False,True,False
249997,33,False,False,False
249998,2629,False,True,False


In [27]:
test_pred = model.predict(submit_x)
test_pred

array([0.08592661, 0.03576086, 0.0499545 , ..., 0.06223786, 0.11807539,
       0.15930685], shape=(250000,))

In [28]:
submission = pd.DataFrame({
    'id': df_test['id'],
    'y': test_pred
})
submission.to_csv('submission.csv', index=False)